In [124]:
# Import the basic libraries to build our own neural network model
import numpy as np
import pandas as pd

In [125]:
# Initialize simple paramters of same scale and if possible small values so the model converges faster 
df = pd.DataFrame(
                    [
                        [8,8,4],
                        [7,9,5],
                        [6,10,6],
                        [5,12,7]
                    ]
                    , columns = ['cgpa', 'profile_score', 'lpa']
                )

In [126]:
df # Check the dataset

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


In [127]:
# A utility function to initialize values to the parameter based on the list passed
# The passed list contains : pair of each matrix dimensions for the weights and bias

def initialize_parameters(layer_dims):

  np.random.seed(3)
  parameters = {}
  L = len(layer_dims) # This is basically the number of layers in the neural network

  for l in range(1, L): # We start with 1 because we need it 

    # print(f'\n Layer : {l}\n')

    parameters['W' + str(l)] = np.ones(
                                        (layer_dims[l-1], layer_dims[l]) # The shape of weight matrix for layer l
                                      )*0.1
    
    parameters['b' + str(l)] = np.zeros(
                                          (layer_dims[l], 1) # The shape of bias matrix for layer l
                                        )

    # print(f'\n Weights : \n {parameters['W' + str(l)]} \n ')
    # print(f'\n Bias : \n {parameters['b' + str(l)]} \n ')

  return parameters

In [128]:
parameters = initialize_parameters([2,2,1])
# This means that the first weight matrix will be having layer_dims[1-1] = 2 rows and layer_dims[1] =2 columns
# No of bias is equal to layer_dims[1-1] = 2 rows and 1 column (Fixed)

# Next layer will have weight layer_dims[2-1] = 2 rows and layer_dims[2] = 1 column
# No of bias is equal to 1 row and 1 column (Fixed)

In [129]:
len(parameters) // 2 # <-- No of layers in neural network

2

In [130]:
# A simple function that passess the linear dot product of weights and Inputs from previous neuron wih bias added to form Z
# It will be used by neurons to calculate the output of them using weights,bias and input

def linear_forward(O_prev, W, b):

  # Z = np.dot(transpose_of_weight_matrix,input_to_neuron) + bias

  Z = np.dot(W.T, O_prev) + b

  return Z

In [131]:
# Forward Propogation using parameters that are made from the utiltiy function before and the matrix X as input

def L_layer_forward(X, parameters):

  A = X # Input to Input layer 1
  L = len(parameters) // 2 # number of layers in the neural network

  for l in range(1, L+1): # An additional loop to calculate the final output

    A_prev = A # In the current loop the previous loop's value becomes the previous_output

    # Fetching the parameters from the paramters hashmap using the layer index
    Wl = parameters['W' + str(l)]
    bl = parameters['b' + str(l)]

    # Pass the matrices to the linear forward utility function so it returns the output(Z)
    A = linear_forward(A_prev, Wl, bl)


  return A,A_prev # In the end the output will be returned with the output of the previous layer 

In [132]:
# Extract a list of row from the data as X and y to evaluate the model
X = df[['cgpa', 'profile_score']].values[0].reshape(2,1) # Shape(no of features, no. of training example)
y = df[['lpa']].values[0][0]

In [133]:
# Parameter initialization with the initialization function
parameters = initialize_parameters([2,2,1])

# Also pass the X tuple here at first to understand how the forward propogation works for a single row
y_hat,A1 = L_layer_forward(X, parameters)

In [134]:
df.iloc[0,:]

cgpa             8
profile_score    8
lpa              4
Name: 0, dtype: int64

In [135]:
y_hat # <-- The prediction for the following above row

array([[0.32]])

In [136]:
A1 # <-- Output of the previous layer

array([[1.6],
       [1.6]])

In [137]:
# This is basically the function which automatically updates all the parameters based on the loss
# The values here are calculated by taking gradient of the parameters and subtracting them with the older values with learning rate to improve model

def update_parameters(parameters, y, y_hat, A1, X, lr=0.001):

  # 1. Store the original W2 values before they are modified
  # This is a must step without this we will be using the newly modified parameters instead which could lead to model updating incorrectly
  w2_0_old = parameters['W2'][0][0]
  w2_1_old = parameters['W2'][1][0]
  
  # 2. Calculate the common error term (assuming MSE loss gradient)
 
  error_term = lr * 2 *  (y_hat - y)

  # 3. Update Output Layer (W2, b2) using old values
  parameters['W2'][0][0] = parameters['W2'][0][0] - (error_term * A1[0][0])
  parameters['W2'][1][0] = parameters['W2'][1][0] - (error_term * A1[1][0])
  parameters['b2'][0][0] = parameters['b2'][0][0] - error_term  # Fixed typo here

  # 4. Update Hidden Layer (W1, b1) using original W2 weights
  parameters['W1'][0][0] = parameters['W1'][0][0] - (error_term * w2_0_old * X[0][0])
  parameters['W1'][0][1] = parameters['W1'][0][1] - (error_term * w2_0_old * X[1][0])
  parameters['b1'][0][0] = parameters['b1'][0][0] - (error_term * w2_0_old)

  parameters['W1'][1][0] = parameters['W1'][1][0] - (error_term * w2_1_old * X[0][0])
  parameters['W1'][1][1] = parameters['W1'][1][1] - (error_term * w2_1_old * X[1][0])
  parameters['b1'][1][0] = parameters['b1'][1][0] - (error_term * w2_1_old)


In [138]:
# Now lets apply this to the row one by one and then make prediction again 

In [139]:
# so the current parameters are as seen because we have not updated it yet according to loss of row 1
parameters

{'W1': array([[0.1, 0.1],
        [0.1, 0.1]]),
 'b1': array([[0.],
        [0.]]),
 'W2': array([[0.1],
        [0.1]]),
 'b2': array([[0.]])}

In [141]:
y_hat = y_hat[0][0]  # convert to scalar first

In [142]:
# Now we will pass the arguments in the update_parameter function to update the parameters
update_parameters(
                    parameters,
                    y, # The output of the row
                    y_hat, # The prediction done by model,
                    A1, # The second last output from that layer
                    X # The row that was passed
)

In [143]:
# After updating the parameters become :
parameters

{'W1': array([[0.105888, 0.105888],
        [0.105888, 0.105888]]),
 'b1': array([[0.000736],
        [0.000736]]),
 'W2': array([[0.111776],
        [0.111776]]),
 'b2': array([[0.00736]])}

In [148]:
# Row 1

X = df[['cgpa', 'profile_score']].values[0].reshape(2,1) # Shape(no of features, no. of training example)
y = df[['lpa']].values[0][0]

# Parameter initialization
parameters = initialize_parameters([2,2,1])

print(f'\n Paramaters before : \n {parameters} \n')

y_hat,A1 = L_layer_forward(X,parameters)
y_hat = y_hat[0][0]

update_parameters(parameters,y,y_hat,A1,X)

print(f'\n Paramaters After : \n {parameters} \n')


 Paramaters before : 
 {'W1': array([[0.1, 0.1],
       [0.1, 0.1]]), 'b1': array([[0.],
       [0.]]), 'W2': array([[0.1],
       [0.1]]), 'b2': array([[0.]])} 


 Paramaters After : 
 {'W1': array([[0.105888, 0.105888],
       [0.105888, 0.105888]]), 'b1': array([[0.000736],
       [0.000736]]), 'W2': array([[0.111776],
       [0.111776]]), 'b2': array([[0.00736]])} 



In [149]:
# Row 2

X = df[['cgpa', 'profile_score']].values[1].reshape(2,1) # Shape(no of features, no. of training exaplme)
y = df[['lpa']].values[1][0]

print(f'\n Paramaters before : \n {parameters} \n')

y_hat,A1 = L_layer_forward(X,parameters)
y_hat = y_hat[0][0]

update_parameters(parameters,y,y_hat,A1,X)

print(f'\n Paramaters After : \n {parameters} \n')


 Paramaters before : 
 {'W1': array([[0.105888, 0.105888],
       [0.105888, 0.105888]]), 'b1': array([[0.000736],
       [0.000736]]), 'W2': array([[0.111776],
       [0.111776]]), 'b2': array([[0.00736]])} 


 Paramaters After : 
 {'W1': array([[0.11310786, 0.11517068],
       [0.11310786, 0.11517068]]), 'b1': array([[0.00176741],
       [0.00176741]]), 'W2': array([[0.12741603],
       [0.12741603]]), 'b2': array([[0.01658746]])} 



In [150]:
# Row 3

X = df[['cgpa', 'profile_score']].values[2].reshape(2,1) # Shape(no of features, no. of training exaplme)
y = df[['lpa']].values[2][0]

print(f'\n Paramaters before : \n {parameters} \n')

y_hat,A1 = L_layer_forward(X,parameters)
y_hat = y_hat[0][0]

update_parameters(parameters,y,y_hat,A1,X)

print(f'\n Paramaters After : \n {parameters} \n')


 Paramaters before : 
 {'W1': array([[0.11310786, 0.11517068],
       [0.11310786, 0.11517068]]), 'b1': array([[0.00176741],
       [0.00176741]]), 'W2': array([[0.12741603],
       [0.12741603]]), 'b2': array([[0.01658746]])} 


 Paramaters After : 
 {'W1': array([[0.1215442 , 0.12923125],
       [0.1215442 , 0.12923125]]), 'b1': array([[0.00317347],
       [0.00317347]]), 'W2': array([[0.14740615],
       [0.14777037]]), 'b2': array([[0.02762262]])} 



In [151]:
# Row 4

X = df[['cgpa', 'profile_score']].values[3].reshape(2,1) # Shape(no of features, no. of training exaplme)
y = df[['lpa']].values[3][0]

print(f'\n Paramaters before : \n {parameters} \n')

y_hat,A1 = L_layer_forward(X,parameters)
y_hat = y_hat[0][0]

update_parameters(parameters,y,y_hat,A1,X)

print(f'\n Paramaters After : \n {parameters} \n')


 Paramaters before : 
 {'W1': array([[0.1215442 , 0.12923125],
       [0.1215442 , 0.12923125]]), 'b1': array([[0.00317347],
       [0.00317347]]), 'W2': array([[0.14740615],
       [0.14777037]]), 'b2': array([[0.02762262]])} 


 Paramaters After : 
 {'W1': array([[0.13089303, 0.15166842],
       [0.13091613, 0.15172386]]), 'b1': array([[0.00504323],
       [0.00504785]]), 'W2': array([[0.17365565],
       [0.17567747]]), 'b2': array([[0.04030707]])} 



In [152]:
# This was a complete loop of updating parameters row by row !!!
# This was just a single epoch !

In [ ]:
# epochs implementation

parameters = initialize_parameters([2,2,1])
epochs = 10 # Now the whole process is done this much times till model learns on the data

# Traverse # epochs
for i in range(epochs):

  Loss = []

  # Traverse the index of row for each epoch
  for j in range(df.shape[0]):

    X = df[['cgpa', 'profile_score']].values[j].reshape(2,1) # Shape(no of features, no. of training example)
    y = df[['lpa']].values[j][0] # Make sure that both the row and column is specified to make sure the actual value is passed instead of dataframe

    # Parameter initialization
    y_hat,A1 = L_layer_forward(X,parameters) # Pass to the forward propogation function
    y_hat = y_hat[0][0] # Important to make sure we are passing the value and not dataframe

    # Update parameters
    update_parameters(parameters,y,y_hat,A1,X)

    Loss.append((y-y_hat)**2)

  print('Epoch - ',i+1,'Loss - ',np.array(Loss).mean())

Epoch -  1 Loss -  26.37409659194819
Epoch -  2 Loss -  20.072736010883336
Epoch -  3 Loss -  11.300115717590234
Epoch -  4 Loss -  4.160816947552781
Epoch -  5 Loss -  1.49698118186726
Epoch -  6 Loss -  1.1592901306389134
Epoch -  7 Loss -  1.214161713757349
Epoch -  8 Loss -  1.2554327317325071
Epoch -  9 Loss -  1.2714893175557096
Epoch -  10 Loss -  1.2775007670089384


In [163]:
parameters # <-- Final parameters

{'W1': array([[0.25488858, 0.42543315],
        [0.26360616, 0.45264698]]),
 'b1': array([[0.02901186],
        [0.03114039]]),
 'W2': array([[0.44589875],
        [0.52916829]]),
 'b2': array([[0.12643907]])}

In [166]:
Loss # Loss of final rows 

[np.float64(2.953270572984179),
 np.float64(0.0890264242811666),
 np.float64(0.5755338474026456),
 np.float64(1.4921722233677623)]